# Skill-level × Codeforces experiment on Gemma 4 12B (Colab)

Runs the v2.1 experiment (**does a stated user skill level change the code a model writes?**) on the open-weight
`google/gemma-4-12B-it` model instead of `gpt-oss-20b` via Groq, so there is no daily token cap.

Design is identical to the Groq run: 24 Codeforces problems × (4 skill levels × 2 disclosure channels + control) × 3 repeats = **648 answers**,
byte-identical task text across conditions, fixed shuffle seed, same sandboxed stdin/stdout scorer and the same pre-registered statistics.

**Hardware.** The model is ~12B parameters. In bf16 it needs ~24 GB of weights plus KV cache, so use an **A100 (40/80 GB)** runtime (Colab Pro).
On an L4 (24 GB) or T4 (15 GB) the notebook falls back to 4-bit quantisation, which is slower and is a slightly different model; a T4 run of all 648 answers takes well over the free 12-hour session limit, so reduce `REPS` there.
Progress is checkpointed to Google Drive after every batch and the notebook resumes where it left off.

**Before running:** Runtime → Change runtime type → GPU. Add a Colab secret named `HF_TOKEN` (Hugging Face token; accept the Gemma licence on the model page first if it asks).\n\n**Getting the task files.** The GitHub repo is private, so cell 4 needs **either** a Colab secret `GITHUB_TOKEN` (a GitHub personal access token with repo read access) **or** the file `colab_bundle.zip` from the repo's `experiment/` folder: put it in `MyDrive/user-model-gemma4/` on Google Drive, or upload it when cell 4 asks.

In [ ]:
#@title 1. Runtime check
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"VRAM: {VRAM_GB:.0f} GB")

In [ ]:
#@title 2. Install libraries (Gemma 4 needs a recent transformers)
%pip -q install -U "transformers>=4.57" accelerate bitsandbytes huggingface_hub
import importlib, transformers
print("transformers", transformers.__version__)
try:
    from transformers import AutoConfig
    AutoConfig.from_pretrained("google/gemma-4-12B-it", token=None)
except Exception as e:  # model type unknown to this release -> install from main
    if "gemma4" in str(e).lower() or "model type" in str(e).lower():
        print("Gemma 4 not in this release; installing transformers from GitHub main...")
        %pip -q install -U git+https://github.com/huggingface/transformers
        print("Now restart the runtime (Runtime -> Restart session) and run from cell 1 again.")
    else:
        print("Config check skipped:", type(e).__name__, str(e)[:200])

In [ ]:
#@title 3. Hugging Face login + Google Drive (for checkpoints)
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise SystemExit("Add a Colab secret named HF_TOKEN (Hugging Face token) and re-run.")
login(token=HF_TOKEN, add_to_git_credential=False)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/user-model-gemma4"
except Exception:
    SAVE_DIR = "/content/user-model-gemma4"
os.makedirs(SAVE_DIR, exist_ok=True)
print("checkpoints ->", SAVE_DIR)

In [ ]:
#@title 4. Get the experiment code and tasks (private repo: token clone, or upload the bundle)
import os, subprocess, zipfile, json
EXP = "/content/user-model/experiment"
REPO = "github.com/cby437/user-model.git"
BRANCH = "claude/education-level-genai-experiment-27e674"
NEEDED = ["tasks_cf.json", "conditions_skill.json", "score.py", "score_cf.py", "stats.py"]

def have_all():
    return all(os.path.exists(os.path.join(EXP, f)) for f in NEEDED)

# Option A: clone with a GitHub token stored as a Colab secret GITHUB_TOKEN (repo is private)
if not have_all():
    try:
        from google.colab import userdata
        GH = userdata.get("GITHUB_TOKEN")
    except Exception:
        GH = os.environ.get("GITHUB_TOKEN")
    if GH:
        r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://{GH}@{REPO}", "/content/user-model"],
                           capture_output=True, text=True)
        print("clone:", "ok" if r.returncode == 0 else r.stderr[-300:].replace(GH, "***"))

# Option B: the bundle zip (colab_bundle.zip) on Drive or uploaded now
if not have_all():
    os.makedirs(EXP, exist_ok=True)
    bundle = os.path.join(SAVE_DIR, "colab_bundle.zip")
    if not os.path.exists(bundle):
        # Accept the bundle anywhere obvious on Drive, under any spelling like colab.bundle.zip
        import glob
        found = [p for d in (SAVE_DIR, "/content/drive/MyDrive", "/content")
                 for p in glob.glob(os.path.join(d, "colab*bundle*.zip"))]
        if found:
            bundle = found[0]
            print("using bundle:", bundle)
    if not os.path.exists(bundle):
        print("No GITHUB_TOKEN secret and no bundle on Drive. Upload colab_bundle.zip now:")
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        import shutil
        try:
            shutil.move(name, bundle)        # keep a copy on Drive for next time
        except Exception as e:               # Drive not writable: use it where it landed
            print("could not copy to Drive:", type(e).__name__, "- using local copy")
            bundle = os.path.abspath(name)
    with zipfile.ZipFile(bundle) as z:
        z.extractall(EXP)
    print("bundle extracted ->", EXP)

assert have_all(), "missing files: " + ", ".join(f for f in NEEDED if not os.path.exists(os.path.join(EXP, f)))
os.chdir(EXP)
os.makedirs("out", exist_ok=True)
# Prefer the 24-task battery selected by the control screening pilot; fall back to the full 44.
TASK_FILE = "tasks_cf_selected.json" if os.path.exists("tasks_cf_selected.json") else "tasks_cf.json"
print("task file:", TASK_FILE, "|", len(json.load(open(TASK_FILE))), "tasks")

In [ ]:
#@title 5. Settings (match the Groq run unless you have a reason not to)
MODEL_ID = "google/gemma-4-12B-it"   # instruction-tuned chat variant of google/gemma-4-12B
REPS = 3                              # 3 -> 648 answers on 24 tasks; use 1 on a T4
MAX_NEW_TOKENS = 3000                 # same answer budget as the Groq run
TEMPERATURE = 0.7                     # same as the Groq run (Gemma's card suggests 1.0)
TOP_P, TOP_K = 0.95, 64               # Gemma 4 recommended
THINKING = False                      # Gemma 4 hidden reasoning off: fixed budget goes to the answer
SHUFFLE_SEED = 20261014               # same shuffle as the Groq run
OUT_JSONL = os.path.join(SAVE_DIR, f"responses_cf_gemma4.jsonl")
import torch
if VRAM_GB >= 36:
    LOAD_MODE, BATCH = "bf16", 8
elif VRAM_GB >= 20:
    LOAD_MODE, BATCH = "4bit", 4
else:
    LOAD_MODE, BATCH = "4bit", 2
print(f"load={LOAD_MODE} batch={BATCH} reps={REPS}")

In [ ]:
#@title 6. Build the condition matrix (same wording as conditions_skill.json; task text byte-identical)
import json, random
tasks = json.load(open(TASK_FILE))
cond = json.load(open("conditions_skill.json"))

def build_cells():
    for t in tasks:
        meta = {"task_id": t["id"], "language": t["language"], "difficulty": t["difficulty"],
                "hardening": t["hardening"], "entry_point": t["entry_point"]}
        c = cond["control"]
        yield (f"{t['id']}_control_none", {**meta, "skill_level": "control", "channel": "none"},
               c["system_template"], c["message_template"].format(task=t["prompt"]))
        for lk, ldesc in cond["skill_levels"].items():
            for ck, ch in cond["channels"].items():
                yield (f"{t['id']}_{lk}_{ck}", {**meta, "skill_level": lk, "channel": ck},
                       ch["system_template"].format(level=ldesc),
                       ch["message_template"].format(level=ldesc, task=t["prompt"]))

units = [(f"{p}_r{r}", m, s, u, r) for p, m, s, u in build_cells() for r in range(1, REPS + 1)]
random.Random(SHUFFLE_SEED).shuffle(units)
# byte-identical task text across conditions (last paragraph is the task's final instruction)
texts = {}
for _, m, _, u, _ in units:
    texts.setdefault(m["task_id"], set()).add(u.split("\n\n")[-1])
assert all(len(v) == 1 for v in texts.values()), "task text differs across conditions"
print(len(units), "requests;", len(tasks), "tasks x 9 conditions x", REPS, "reps")

In [ ]:
#@title 7. Load Gemma 4
import torch, transformers
from transformers import AutoProcessor
load_kwargs = {"token": HF_TOKEN, "device_map": "auto"}
if LOAD_MODE == "bf16":
    load_kwargs["dtype"] = torch.bfloat16
else:
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                            bnb_4bit_compute_dtype=torch.bfloat16)
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
tok = getattr(processor, "tokenizer", processor)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = None
for cls_name in ("AutoModelForMultimodalLM", "AutoModelForImageTextToText", "AutoModelForCausalLM"):
    cls = getattr(transformers, cls_name, None)
    if cls is None:
        continue
    try:
        model = cls.from_pretrained(MODEL_ID, **load_kwargs)
        print("loaded with", cls_name)
        break
    except Exception as e:
        print(cls_name, "failed:", str(e)[:160])
assert model is not None, "could not load the model; check transformers version / licence acceptance"
model.eval()
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)

In [ ]:
#@title 8. Generate all answers (checkpointed, resumable)
import json, re, time, os
THOUGHT_RE = re.compile(r"<\|channel>thought\n.*?<channel\|>", re.DOTALL)   # Gemma 4 hidden-reasoning block
SPECIAL_RE = re.compile(r"<\|[^>]*\|>|<end_of_turn>|<start_of_turn>[a-z]*|<eos>|<bos>|<pad>")

def render(system, user):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=THINKING)
    except TypeError:  # template without the enable_thinking switch
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def clean(text):
    text = THOUGHT_RE.sub("", text)
    return SPECIAL_RE.sub("", text).strip()

done = set()
if os.path.exists(OUT_JSONL):
    done = {json.loads(l)["custom_id"] for l in open(OUT_JSONL) if l.strip()}
todo = [u for u in units if u[0] not in done]
todo.sort(key=lambda u: len(u[3]))            # similar prompt lengths per batch -> less padding
print(f"resuming: {len(done)} done, {len(todo)} to go, batch {BATCH}")

eos_ids = [tok.eos_token_id] + [i for i in [tok.convert_tokens_to_ids("<end_of_turn>")] if isinstance(i, int) and i >= 0]
t0 = time.time()
with open(OUT_JSONL, "a") as fh:
    for b in range(0, len(todo), BATCH):
        batch = todo[b:b + BATCH]
        prompts = [render(s, u) for _, _, s, u, _ in batch]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE,
                                 top_p=TOP_P, top_k=TOP_K, eos_token_id=eos_ids, pad_token_id=tok.pad_token_id)
        new = out[:, enc["input_ids"].shape[1]:]
        for (cid, meta, system, user, rep), ids in zip(batch, new):
            ids = ids.tolist()
            n_new = len(ids) - ids.count(tok.pad_token_id) if tok.pad_token_id is not None else len(ids)
            raw = tok.decode(ids, skip_special_tokens=False)
            stop = "stop" if any(e in ids for e in eos_ids) else "length"
            rec = {"custom_id": cid, **meta, "rep": rep, "result_type": "succeeded", "model": MODEL_ID,
                   "temperature": TEMPERATURE, "shuffle_seed": SHUFFLE_SEED, "thinking": THINKING,
                   "load_mode": LOAD_MODE, "response_text": clean(raw), "raw_text": raw,
                   "stop_reason": stop, "output_tokens": n_new}
            fh.write(json.dumps(rec) + "\n")
        fh.flush()
        n = len(done) + b + len(batch)
        el = time.time() - t0
        print(f"  {n}/{len(units)}  ({el/60:.1f} min elapsed, ~{el/(b+len(batch))*(len(todo)-b-len(batch))/60:.0f} min left)")
print("done ->", OUT_JSONL)

In [ ]:
#@title 9. Score (sandboxed stdin/stdout tests) and run the pre-registered statistics
import subprocess, shutil, sys
shutil.copy(OUT_JSONL, "out/responses_cf.jsonl")
print(subprocess.run([sys.executable, "score_cf.py", "--responses", "out/responses_cf.jsonl",
                      "--out", "out/scores_cf_gemma4.csv", "--workers", "4"], capture_output=True, text=True).stdout[-400:])
r = subprocess.run([sys.executable, "stats.py", "--scores", "out/scores_cf_gemma4.csv", "--judge", "",
                    "--out", "out/stats_cf_gemma4.md"], capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-800:])

In [ ]:
#@title 10. Plain summary by skill level
import csv, statistics, collections
rows = list(csv.DictReader(open("out/scores_cf_gemma4.csv")))
levels = ["control", "beginner", "hobbyist", "professional", "expert"]
def m(rs, k, f=float): 
    v = [f(r[k]) for r in rs if r.get(k) not in ("", None)]
    return statistics.mean(v) if v else float("nan")
print(f"{'level':<13}{'n':>5}{'pass':>7}{'all':>7}{'giveup':>8}{'trunc':>7}{'words':>7}{'loc':>6}{'idioms':>8}{'comments':>10}")
for lv in levels:
    rs = [r for r in rows if r["skill_level"] == lv]
    print(f"{lv:<13}{len(rs):>5}{m(rs,'pass_rate'):>7.3f}{m(rs,'all_pass'):>7.3f}{m(rs,'gave_up'):>8.3f}{m(rs,'truncated'):>7.3f}"
          f"{m(rs,'prose_words'):>7.0f}{m(rs,'loc'):>6.0f}{m(rs,'advanced_idioms'):>8.2f}{m(rs,'comment_ratio'):>10.3f}")
print("\nby channel (self-disclosed vs hidden profile):")
for ch in ["message", "system"]:
    for lv in ["beginner", "expert"]:
        rs = [r for r in rows if r["skill_level"] == lv and r["channel"] == ch]
        print(f"  {ch:<8}{lv:<12} pass={m(rs,'pass_rate'):.3f} giveup={m(rs,'gave_up'):.3f} words={m(rs,'prose_words'):.0f}")
shutil.copy("out/scores_cf_gemma4.csv", SAVE_DIR); shutil.copy("out/stats_cf_gemma4.md", SAVE_DIR)
print("\nsaved scores + stats to", SAVE_DIR)